## MLP Classifier Testbench


This notebook is designed to test the implementation of matrix multiplication in the MLP classifier. The model architecture is provided, and you are required to complete the specified sections to enable the optimized matrix multiplication IP of the fully connected layer to run on the programmable logic, in order to evaluate the performance of the entire model.

In [1]:
#@@ Uncomment the following lines when deploying on the board
# from pynq import Overlay
# from pynq import allocate
# from pynq import MMIO
import numpy as np
import time
import matplotlib.pyplot as plt

####### DO NOT CHANGE
IMAGE_SIZE = 28 * 28    # MNIST images are 28x28 pixels
HIDDEN_SIZE1 = 1024     # Neurons in the first hidden layer
HIDDEN_SIZE2 = 512      # Neurons in the second hidden layer
NUM_CLASSES = 10        # Digits 0-9
BATCH_SIZE = 128        # Number of images to process at once
CTRL_ADDR = 0x00        # Control register address
START_VALUE = 1<<0      # bit 0
STOP_VALUE = 0x0        # bit 0
DONE_VALUE = 1<<1       # bit 1
AUTO_RESTART = 1<<7     # bit 7
#######

#@@ Modify addresses according to vivado file
IP_BASE_ADDRESS = 0x4C3000000
ADDRESS_RANGE = 0x10000
A_1_ADDR = 0x10
A_2_ADDR = 0x10
B_1_ADDR = 0x18
B_2_ADDR = 0x18
C_1_ADDR = 0x1C
C_2_ADDR = 0x1C
ABC_1_ADDR = 0x20
ABC_2_ADDR = 0x20
N_ADDR = 0x30
M_ADDR = 0x30
P_ADDR = 0x30

## Variable Declaration

In [2]:
error=np.zeros(BATCH_SIZE)
ind=np.arange(BATCH_SIZE)

input = np.zeros(BATCH_SIZE*IMAGE_SIZE, dtype=np.int32)
w1 = np.zeros(HIDDEN_SIZE1*IMAGE_SIZE, dtype=np.int32)
b1 = np.zeros(HIDDEN_SIZE1, dtype=np.int32)
w2 = np.zeros(HIDDEN_SIZE2*HIDDEN_SIZE1, dtype=np.int32)
b2 = np.zeros(HIDDEN_SIZE2, dtype=np.int32)
w3 = np.zeros(NUM_CLASSES*HIDDEN_SIZE2, dtype=np.int32)
b3 = np.zeros(NUM_CLASSES, dtype=np.int32)
expected_label = np.zeros(BATCH_SIZE, dtype=np.int32)

## Function Definition 

In [3]:
# Function to compare two files line by line
def compare_files(file1, file2):
    with open(file1, 'r') as f1, open(file2, 'r') as f2:
        lines1 = [line.strip() for line in f1]
        lines2 = [line.strip() for line in f2]
    return lines1 == lines2

# Function to read weights and biases from a file
def read_weights_and_biases(wfilename, weight, bias, H, W):
    print(f"Reading Weights and Biases from file: {wfilename}")

    try:
        with open(wfilename, 'r') as wfile:
            data = wfile.read()
    except IOError:
        print(f"Error opening file: {wfilename}")
        return

    weight_index = 0
    bias_index = 0
    buf = ''
    reading_weights = False
    reading_biases = False
    is_negative = False
    i = 0

    while i < len(data):
        ch = data[i]

        if ch == 'W' and data[i:i+2] == 'We':
            reading_weights = True
            reading_biases = False
            # Skip until ']'
            i = data.find(']', i)
            if i == -1:
                break
        elif ch == 'b' and data[i:i+2] == 'bi':
            reading_weights = False
            reading_biases = True
            # Skip until ']'
            i = data.find(']', i)
            if i == -1:
                break

        if i >= len(data):
            break

        ch = data[i]
        if ch.isdigit() or ch == '.' or ch == '-':
            if ch == '-':
                is_negative = True
            else:
                buf += ch
        elif buf:
            try:
                value = int(buf)
                if is_negative:
                    value = -value
                if reading_weights and weight_index < H * W:
                    weight[weight_index] = value
                    weight_index += 1
                elif reading_biases and bias_index < H:
                    bias[bias_index] = value
                    bias_index += 1
            except ValueError:
                pass  # ignore malformed numbers
            buf = ''
            is_negative = False

        i += 1

    print(f"Number of read values for weights: {weight_index}")
    print(f"Number of read values for biases: {bias_index}")
    
# Function to read images and expected labels from a file
def read_image_and_labels(img_filename, lbl_filename, data, label):
    print(f"Reading in images and labels from file: {img_filename} and {lbl_filename}")

    try:
        with open(img_filename, 'r') as img_file, open(lbl_filename, 'r') as lbl_file:
            img_values = list(map(int, img_file.read().split()))
            lbl_values = list(map(int, lbl_file.read().split()))

            if len(img_values) < BATCH_SIZE * IMAGE_SIZE:
                raise ValueError("Not enough image data in file")
            if len(lbl_values) < BATCH_SIZE:
                raise ValueError("Not enough labels in file")

            # Populate the data array with correct layout
            for j in range(BATCH_SIZE):
                for i in range(IMAGE_SIZE):
                    data[i * BATCH_SIZE + j] = img_values[j * IMAGE_SIZE + i] / 256

            # Populate labels
            for j in range(BATCH_SIZE):
                label[j] = lbl_values[j]

    except FileNotFoundError as e:
        print(f"Error opening file: {e.filename}")
    except EOFError as e:
        print(f"Error: {e}")
        exit(1)


## Software Implementation on PYNQ Processing System (PS) (Section 5)

In [4]:
#@@ Complete the fully_connected function to perform matrix multiplication and add biases as a reference to verify the hardware implementation
#@@ Write your code for section 5 here following the instructions in the lab guide

def slow_fully_connected_sw(weights, ins, biases, N, M, P):
    #pass #@@ Implement the fully connected layer operation here
    out = np.zeros((N, P), dtype = np.float32)
    for i in range(N):              
        for j in range(M):
            acc = 0
            for k in range(P):
                acc += weights[i][k] * ins[k][j] 
                out[i][j] = acc + biases[i]
    return out


def fully_connected_sw(weights, ins, biases, N, M, P):
    # Ensure NumPy arrays
    weights = np.array(weights)
    ins = np.array(ins)
    biases = np.array(biases)

    # Matrix multiplication: (N x M) @ (M x P) -> (N x P)
    out = np.dot(weights, ins)

    # Add bias to every column of the output
    out += biases.reshape(N, 1)

    return out


In [ ]:
#test
size_vec = [128, 256, 512, 1024, 2048, 4096]
for s in size_vec:
    N = M = P = s
    weights = np.random.randint(1, 5, size=(N, M))
    ins = np.random.randint(1, 5, size=(M, P))
    biases = np.random.randint(1, 5, size=(N,))

    start = time.time()
    out = fully_connected_sw(weights, ins, biases, N, M, P)
    end = time.time()
    
    golden = weights @ ins + biases[:, None]
    correct = np.allclose(out, golden)
    print(f"Size: {s}, execution time: {end-start}s, Correct: {correct}")

Size: 128, execution time: 0.0023059844970703125s, Correct: True
Size: 256, execution time: 0.006940364837646484s, Correct: True
Size: 512, execution time: 0.19862580299377441s, Correct: True
Size: 1024, execution time: 6.918506860733032s, Correct: True


## Optional : In case you want to verify your Vitis design layer by layer
You are required to write the output of each previous layer to a file, which can then be read in your Vitis testbench.

In [ ]:
# Verify the functionality by comparing to the golden output of first layer, provided in the file "first_layer_golden_output.txt"
read_image_and_labels("mnist_images_int.txt", "mnist_labels.txt", input, expected_label)
print("---------------------------------------------------------------------------------------")
read_weights_and_biases("weights_layer1.txt", w1, b1, HIDDEN_SIZE1, IMAGE_SIZE)
# read_weights_and_biases("weights_layer2.txt", w2, b2, HIDDEN_SIZE2, HIDDEN_SIZE1)
# read_weights_and_biases("weights_layer3.txt", w3, b3, NUM_CLASSES, HIDDEN_SIZE2)
print("---------------------------------------------------------------------------------------")

#@@ Uncomment the corresponding line to generate golden output for a specific layer for functionality verification in Vitis
y1 = fully_connected_sw(w1, input, b1, HIDDEN_SIZE1, IMAGE_SIZE, BATCH_SIZE)
# y2 = fully_connected_sw(w2, y1, b2, HIDDEN_SIZE2, HIDDEN_SIZE1, BATCH_SIZE)
# y3 = fully_connected(w3, y2, b3, NUM_CLASSES, HIDDEN_SIZE2, BATCH_SIZE)
with open("first_layer_actual_output.txt", "w") as f:
    for i in range(HIDDEN_SIZE1*BATCH_SIZE):
        f.write(f"{y1[i]}\n")

if compare_files("first_layer_actual_output.txt", "first_layer_golden_output.txt") != 1:
    print("*******************************************")
    print("FAIL: Output DOES NOT match the golden output")
    print("*******************************************")
    exit(1)
else:
    print("*******************************************")
    print("PASS: The output matches the golden output!")
    print("*******************************************")
    exit(0)

## MLP Activation Functions

In [7]:
def relu(inputs):
    return np.maximum(0, inputs)

def softmax(inputs):
    max = 0
    inputs_exp = np.array([0]*len(inputs))
    output = np.array([0]*len(inputs))
    inputs_exp_sum = 0
    for i in range(len(inputs)):
        if inputs[i] > max:
            max = inputs[i]
    for i in range(len(inputs)):
        inputs_exp[i] = np.exp((inputs[i]/max)-1)
        inputs_exp_sum = inputs_exp_sum + inputs_exp[i]
    for i in range(len(inputs)):
        output[i] = inputs_exp[i] / inputs_exp_sum
    return output

## Loading Hardware IP

In [ ]:
#@@ Change name of bitstream as required
ol=Overlay('./mm.bit')
mm_ip=MMIO(IP_BASE_ADDRESS, ADDRESS_RANGE)

## Deploying MLP on PYNQ

In [ ]:
# Reading weights and biases from files
read_image_and_labels("mnist_images_int.txt", "mnist_labels.txt", input, expected_label)
print("---------------------------------------------------------------------------------------")
read_weights_and_biases("weights_layer1.txt", w1, b1, HIDDEN_SIZE1, BATCH_SIZE)
print("---------------------------------------------------------------------------------------")
read_weights_and_biases("weights_layer2.txt", w2, b2, HIDDEN_SIZE2, HIDDEN_SIZE1)
print("---------------------------------------------------------------------------------------")
read_weights_and_biases("weights_layer3.txt", w3, b3, NUM_CLASSES, HIDDEN_SIZE2)
print("---------------------------------------------------------------------------------------")

start_time = time.time()

#################################################################################################
#@@ MM IP for Fully Connected Layer 1 with y1 as the output
#################################################################################################
#@@ Writing values to the FPGA
# Allocate buffers for input and output data
in1_buf = allocate(shape=(HIDDEN_SIZE1*IMAGE_SIZE,), dtype=np.int32)
in2_buf = allocate(shape=(IMAGE_SIZE*BATCH_SIZE,), dtype=np.int32)
in3_buf = allocate(shape=(HIDDEN_SIZE1,), dtype=np.int32)
out_buf = allocate(shape=(HIDDEN_SIZE1*BATCH_SIZE,), dtype=np.int32)
# Copy data to the buffers
np.copyto(in1_buf, np.int32(w1))
np.copyto(in2_buf, np.int32(input))
np.copyto(in3_buf, np.int32(b1))
# Write pointer values to sepicified addresses of the IP
mm_ip.write(A_1_ADDR, in1_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(B_1_ADDR, in2_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(C_1_ADDR, in3_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(ABC_1_ADDR, out_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(A_2_ADDR, (in1_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(B_2_ADDR, (in2_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(C_2_ADDR, (in3_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(ABC_2_ADDR, (out_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(N_ADDR, HIDDEN_SIZE1)
mm_ip.write(M_ADDR, IMAGE_SIZE)
mm_ip.write(P_ADDR, BATCH_SIZE)

#Starting and stopping the IP (Don't change this)
mm_ip.write(CTRL_ADDR, START_VALUE)
# Wait until done
# while (mm_ip.read(CTRL_ADDR) & DONE_VALUE) == 0:
#     pass
while True:
    if mm_ip.read(CTRL_ADDR) != START_VALUE:
        break
mm_ip.write(CTRL_ADDR, STOP_VALUE)

#@@ Reading from IP
y1 = out_buf
y1 = relu(y1)

#################################################################################################
#@@ MM IP for Fully Connected Layer 2 with y2 as the output
#################################################################################################
#@@ Writing values to the FPGA
# Allocate buffers for input and output data
in1_buf = allocate(shape=(HIDDEN_SIZE2*HIDDEN_SIZE1,), dtype=np.int32)
in2_buf = allocate(shape=(HIDDEN_SIZE1*BATCH_SIZE,), dtype=np.int32)
in3_buf = allocate(shape=(HIDDEN_SIZE2,), dtype=np.int32)
out_buf = allocate(shape=(HIDDEN_SIZE2*BATCH_SIZE,), dtype=np.int32)
# Copy data to the buffers
np.copyto(in1_buf, np.int32(w2))
np.copyto(in2_buf, np.int32(y1))
np.copyto(in3_buf, np.int32(b2))
# Write pointer values to sepicified addresses of the IP
mm_ip.write(A_1_ADDR, in1_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(B_1_ADDR, in2_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(C_1_ADDR, in3_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(ABC_1_ADDR, out_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(A_2_ADDR, (in1_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(B_2_ADDR, (in2_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(C_2_ADDR, (in3_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(ABC_2_ADDR, (out_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(N_ADDR, HIDDEN_SIZE2)
mm_ip.write(M_ADDR, HIDDEN_SIZE1)
mm_ip.write(P_ADDR, BATCH_SIZE)

#Starting and stopping the IP (Don't change this)
mm_ip.write(CTRL_ADDR, START_VALUE)
# Wait until done
# while (mm_ip.read(CTRL_ADDR) & DONE_VALUE) == 0:
#     pass
while True:
    if mm_ip.read(CTRL_ADDR) != START_VALUE:
        break
mm_ip.write(CTRL_ADDR, STOP_VALUE)

#@@ Reading from IP
y2 = out_buf
y2 = relu(y2)

#################################################################################################
#@@ MM IP for Fully Connected Layer 3 with y3 as the output
#################################################################################################
#@@ Writing values to the FPGA
# Allocate buffers for input and output data
in1_buf = allocate(shape=(NUM_CLASSES*HIDDEN_SIZE2,), dtype=np.int32)
in2_buf = allocate(shape=(HIDDEN_SIZE2*BATCH_SIZE,), dtype=np.int32)
in3_buf = allocate(shape=(NUM_CLASSES,), dtype=np.int32)
out_buf = allocate(shape=(NUM_CLASSES*BATCH_SIZE,), dtype=np.int32)
# Copy data to the buffers
np.copyto(in1_buf, np.int32(w3))
np.copyto(in2_buf, np.int32(y2))
np.copyto(in3_buf, np.int32(b3))
# Write pointer values to sepicified addresses of the IP
mm_ip.write(A_1_ADDR, in1_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(B_1_ADDR, in2_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(C_1_ADDR, in3_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(ABC_1_ADDR, out_buf.physical_address & 0xFFFFFFFF)
mm_ip.write(A_2_ADDR, (in1_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(B_2_ADDR, (in2_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(C_2_ADDR, (in3_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(ABC_2_ADDR, (out_buf.physical_address >> 32) & 0xFFFFFFFF)
mm_ip.write(N_ADDR, NUM_CLASSES)
mm_ip.write(M_ADDR, HIDDEN_SIZE2)
mm_ip.write(P_ADDR, BATCH_SIZE)

#Starting and stopping the IP (Don't change this)
mm_ip.write(CTRL_ADDR, START_VALUE)
# Wait until done
# while (mm_ip.read(CTRL_ADDR) & DONE_VALUE) == 0:
#     pass
while True:
    if mm_ip.read(CTRL_ADDR) != START_VALUE:
        break
mm_ip.write(CTRL_ADDR, STOP_VALUE)

#@@ Reading from IP
y3 = out_buf
y4 = np.array(y3).reshape(10,128)
for i in range(BATCH_SIZE):
    y4[:, i] = softmax(y4[:, i])
predictions = np.argmax(y4, axis=0)

correct_predictions = 0
correct_predictions += np.sum(predictions == expected_label)

testing_time = (time.time() - start_time) * 1000
accuracy = 100.0 * correct_predictions / BATCH_SIZE
print(f"Testing finished in {testing_time:.2f} miliseconds.")
print(f'Accuracy on the {BATCH_SIZE} test images using manual inference: {accuracy:.2f}%')

Reading input images and labels from file: mnist_images_int.txt and mnist_labels.txt
---------------------------------------------------------------------------------------
Reading Weights and Biases from file: weights_layer1.txt
Number of read values for weights: 131072
Number of read values for biases: 1024
---------------------------------------------------------------------------------------
Reading Weights and Biases from file: weights_layer2.txt
Number of read values for weights: 524288
Number of read values for biases: 512
---------------------------------------------------------------------------------------
Reading Weights and Biases from file: weights_layer3.txt
Number of read values for weights: 5120
Number of read values for biases: 10
---------------------------------------------------------------------------------------


NameError: name 'mm_ip' is not defined